In [3]:
import re
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [5]:
from test_tokenizer import FastWordPieceTokenizer
with open("wikitext2_tokenizer_wordpiece.pkl", "rb") as f:
    tokenizer = pickle.load(f)


In [6]:
print("✅ Tokenizer loaded | Vocab size:", len(tokenizer.vocab))

✅ Tokenizer loaded | Vocab size: 43503


In [7]:

# ==========================================================
# 2️⃣ PARSE WIKITEXT SECTIONS
# ==========================================================
def parse_wikitext(file_path):
    """
    Parse WikiText into coherent sections (no topic overlap).
    Splits at lines starting with '=', '==', or '==='.
    """
    data = []
    current_section = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if re.match(r"^=+", line):  # New topic/subtopic/point
                if current_section:
                    data.append(" ".join(current_section))
                    current_section = []
                continue

            current_section.append(line)

        if current_section:
            data.append(" ".join(current_section))

    print(f"✅ Parsed {len(data)} coherent sections from WikiText.")
    return data


sections = parse_wikitext("wikitext2_train.txt")

✅ Parsed 5500 coherent sections from WikiText.


In [8]:
class WikiTextDataset(Dataset):
    def __init__(self, sections, tokenizer, seq_len=50):
        self.seq_len = seq_len
        self.samples = []
        self.vocab = tokenizer.vocab
        self.unk_token_id = tokenizer.unk_token_id
        self.tokenizer = tokenizer

        for section in sections:
            tokens = [self.vocab.get(tok, self.unk_token_id)
                      for tok in tokenizer.tokenize(section)]

            # Create sequential samples inside the section
            for i in range(0, len(tokens) - seq_len):
                x = tokens[i:i+seq_len]
                y = tokens[i+1:i+seq_len+1]
                self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


In [10]:
dataset = WikiTextDataset(sections, tokenizer, seq_len=40)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
print(f"✅ Dataset ready | Total samples: {len(dataset)}")

✅ Dataset ready | Total samples: 1902735


In [15]:
class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers,dropout=0.3, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, hidden=None):
        embed = self.embedding(x)
        output, hidden = self.gru(embed, hidden)
        out = self.layer_norm(output)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits, hidden 

In [16]:
vocab_size = len(tokenizer.vocab)
model = GRULanguageModel(vocab_size).to("cuda" if torch.cuda.is_available() else "cpu")
device = next(model.parameters()).device

print(f"✅ Model initialized on {device}")

✅ Model initialized on cuda:0


In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

In [23]:
EPOCHS = 20

start_epoch = 0
checkpoint_path = "gru_last.pth"
import os
if os.path.exists(checkpoint_path):
    print("🔁 Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path, map_location=cfg.device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")

In [24]:

# ==========================================================
# 6️⃣ TRAINING LOOP
# ==========================================================
from torch.nn.utils import clip_grad_norm_
        

    


for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0

    progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for x, y in progress:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)

        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    print(f"Epoch {epoch+1} done | Avg Loss: {total_loss/len(dataloader):.4f}")

    scheduler.step()

    # ---------- SAVE CHECKPOINT ----------
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
    }, checkpoint_path)

    print(f"✅ Checkpoint saved at epoch {epoch+1}")

print("🎉 Training Complete!")


Epoch 1/20: 100%|████████████| 14866/14866 [36:54<00:00,  6.71it/s, loss=3.5424]


Epoch 1 done | Avg Loss: 3.9839
✅ Checkpoint saved at epoch 1


Epoch 2/20: 100%|████████████| 14866/14866 [36:28<00:00,  6.79it/s, loss=3.0515]


Epoch 2 done | Avg Loss: 3.3695
✅ Checkpoint saved at epoch 2


Epoch 3/20: 100%|████████████| 14866/14866 [36:32<00:00,  6.78it/s, loss=3.0248]


...

Epoch 18 done | Avg Loss: 1.9685
✅ Checkpoint saved at epoch 18


Epoch 19/20: 100%|███████████| 14866/14866 [36:19<00:00,  6.82it/s, loss=2.1936]


Epoch 19 done | Avg Loss: 1.9852
✅ Checkpoint saved at epoch 19


Epoch 20/20: 100%|███████████| 14866/14866 [36:26<00:00,  6.80it/s, loss=2.0794]


Epoch 20 done | Avg Loss: 2.0094
✅ Checkpoint saved at epoch 20
🎉 Training Complete!


In [29]:
model = GRULanguageModel(vocab_size)
checkpoint = torch.load("gru_last.pth", map_location="cpu")
model.load_state_dict(checkpoint["model"])
model.to("cpu")
model.eval()

print("✅ GRU model loaded successfully.")

✅ GRU model loaded successfully.


In [51]:
VOCAB_SIZE = len(tokenizer.vocab)  # Example: Replace with your actual vocab size
EMBED_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 3
CHECKPOINT_PATH = "gru_last.pth" # Replace with your .pth file path
ONNX_OUTPUT_PATH = "gru_model.onnx"
# ---------------------------------

print("Loading model from checkpoint...")
model = GRULanguageModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)

# **MODIFICATION 1: Ensure checkpoint is loaded to CPU**
# (Your original code already did this, which is correct)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=torch.device('cpu')) 
model.load_state_dict(checkpoint['model'])

# **MODIFICATION 2: Move model to CPU**
model.to('cpu') 
model.eval()
print("Model loaded successfully.")

Loading model from checkpoint...
Model loaded successfully.


In [55]:
%env CUDA_LAUNCH_BLOCKING=1

env: CUDA_LAUNCH_BLOCKING=1


In [56]:
# 4. Create Dummy Inputs on CPU
batch_size = 1
sequence_length = 10

# **MODIFICATION 3: Create dummy tensors on CPU**
dummy_x = torch.randint(0, VOCAB_SIZE, (batch_size, sequence_length), dtype=torch.long).to('cpu')
dummy_hidden = torch.randn(NUM_LAYERS, batch_size, HIDDEN_DIM).to('cpu')

example_args = (dummy_x, dummy_hidden)

# 5. Define Dynamic Shapes (No change needed)
batch_dim = torch.export.Dim("batch_size")
seq_len_dim = torch.export.Dim("sequence_length")
dynamic_shapes = (
    {0: batch_dim, 1: seq_len_dim},  
    {1: batch_dim}                  
)

# 6. Export the Model (The NEW Way)
print(f"Exporting model to {ONNX_OUTPUT_PATH} using Dynamo...")
onnx_program = torch.onnx.export(
    model,
    example_args,                          # Pass the example inputs
    dynamo=True,                           # <-- The key flag
    dynamic_shapes=dynamic_shapes,         # <-- The new argument
    input_names=['input_tokens', 'hidden_in'],  # Still good practice
    output_names=['logits_out', 'hidden_out'], # Still good practice
    opset_version=12
)

# 7. Save the Exported Model
print("Saving ONNX model...")
onnx_program.save(ONNX_OUTPUT_PATH)
print("Export complete.")



W1029 18:55:55.105000 9125 site-packages/torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Exporting model to gru_model.onnx using Dynamo...
[torch.onnx] Obtain model graph for `GRULanguageModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GRULanguageModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ❌


ConversionError: Failed to decompose the FX graph for ONNX compatibility. [96mThis is step 2/3[0m of exporting the model to ONNX. Next steps:
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.
- Create an error report with `torch.onnx.export(..., report=True)`, and save the ExportedProgram as a pt2 file. Create an issue in the PyTorch GitHub repository against the [96m*onnx*[0m component. Attach the error report and the pt2 model.

## Exception summary

<class 'torch.AcceleratorError'>: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable dev

In [46]:
torch.__version__

'2.9.0+cu128'